# MyStake App — Build APK on Google Colab

Builds the Android APK from this repo. **No PC / Android Studio needed.**

**Steps:**
1. Click **Runtime → Run all** (or run each cell top to bottom).
2. Wait ~5–10 minutes on first run (installs Java + Android SDK + Gradle).
3. `app-debug.apk` auto-downloads at the end — send it to your phone and install it (allow **"Install unknown apps"** when asked).

In [ ]:
import os
# --- 1) Java 17 (required by Android Gradle Plugin 8.x) ---
!sudo apt-get update -qq
!sudo apt-get install -y -qq openjdk-17-jdk unzip wget > /dev/null
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
!java -version

In [ ]:
import os
# --- 2) Android SDK command-line tools + packages ---
os.environ['ANDROID_HOME'] = '/opt/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/opt/android-sdk'
os.environ['PATH'] = '/opt/android-sdk/cmdline-tools/latest/bin:/opt/android-sdk/platform-tools:' + os.environ['PATH']

!mkdir -p /opt/android-sdk/cmdline-tools
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdtools.zip
!rm -rf /opt/android-sdk/cmdline-tools/latest /opt/android-sdk/cmdline-tools/cmdline-tools
!unzip -q -o /tmp/cmdtools.zip -d /opt/android-sdk/cmdline-tools
!mv /opt/android-sdk/cmdline-tools/cmdline-tools /opt/android-sdk/cmdline-tools/latest
!yes | sdkmanager --licenses > /dev/null
!sdkmanager "platform-tools" "platforms;android-34" "build-tools;34.0.0"

In [ ]:
import os
# --- 3) Gradle 8.7 (required by AGP 8.5) ---
os.environ['PATH'] = '/opt/gradle-8.7/bin:' + os.environ['PATH']
!wget -q https://services.gradle.org/distributions/gradle-8.7-bin.zip -O /tmp/gradle.zip
!rm -rf /opt/gradle-8.7
!unzip -q -o /tmp/gradle.zip -d /opt
!gradle --version

In [ ]:
# --- 4) Download the source code ---
!rm -rf /content/mystakeapp
!git clone --branch arena/01a0aca7-mystakeapp --depth 1 https://github.com/Mohamed2020p/mystakeapp.git /content/mystakeapp
!ls /content/mystakeapp

In [ ]:
# --- 5) Build the debug APK (installable directly) ---
!cd /content/mystakeapp && gradle assembleDebug --no-daemon

In [ ]:
# --- 6) Download the APK ---
!ls -lh /content/mystakeapp/app/build/outputs/apk/debug/
from google.colab import files
files.download('/content/mystakeapp/app/build/outputs/apk/debug/app-debug.apk')

## Optional: signed Release APK

Run the cells below to build a signed release APK. Change `mystake123` to your own passwords, and **back up `release.keystore`** — if you lose it you can never update the app under the same signature.

In [ ]:
import os
os.environ['PATH'] = '/opt/gradle-8.7/bin:/opt/android-sdk/build-tools/34.0.0:' + os.environ['PATH']
# --- 7a) Create signing key (only if missing) ---
!test -f /content/mystakeapp/release.keystore || keytool -genkeypair -v -keystore /content/mystakeapp/release.keystore -alias mystake -keyalg RSA -keysize 2048 -validity 10000 -storepass mystake123 -keypass mystake123 -dname "CN=MyStake App" 2>&1 | tail -3
# --- 7b) Build unsigned release ---
!cd /content/mystakeapp && gradle assembleRelease --no-daemon
# --- 7c) Align + sign + verify ---
!zipalign -v -p 4 /content/mystakeapp/app/build/outputs/apk/release/app-release-unsigned.apk /content/mystakeapp/app-release-aligned.apk 2>&1 | tail -2
!apksigner sign --ks /content/mystakeapp/release.keystore --ks-pass:pass:mystake123 --key-pass:pass:mystake123 --out /content/mystakeapp/app-release.apk /content/mystakeapp/app-release-aligned.apk
!apksigner verify --print-certs /content/mystakeapp/app-release.apk 2>&1 | head -5
!ls -lh /content/mystakeapp/app-release.apk

In [ ]:
# --- 7d) Download release APK + BACK UP the keystore ---
from google.colab import files
files.download('/content/mystakeapp/app-release.apk')
files.download('/content/mystakeapp/release.keystore')